> 在线讲义：[第 5 章 · VLM：图片切块和文字一起排队进 Transformer](https://wikieden.github.io/vla-from-scratch/lessons/05-vlm.html) · 本书《VLA 从零到 π0》

# 第 5 章 · VLM：图片切块和文字一起排队进 Transformer

配套讲义：仓库里 `vla-deep-dive/notes/lessons/05-vlm.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python + NumPy，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 写法 | 跑出来 |
|---|---|---|
| 第 1 步 | `num_patches(224, 14)` | 256；3 个相机 768 |
| 第 2 步 | `prefix_lm_mask(7, 3)` | 前 7 行前 7 列全 1；后 3 行下三角 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np

**函数 `num_patches`**：一张 image × image 的图，切成 patch × patch 的块，一共多少块。

In [ ]:
def num_patches(image, patch):
    """一张 image × image 的图，切成 patch × patch 的块，一共多少块。"""
    return (image // patch) ** 2

**函数 `prefix_lm_mask`**：prefix-LM 遮挡矩阵：行 = 谁在看，列 = 被看的；前缀互相全看，回答只能看前面。

In [ ]:
def prefix_lm_mask(n_prefix, n_suffix):
    """prefix-LM 遮挡矩阵：行 = 谁在看，列 = 被看的；前缀互相全看，回答只能看前面。"""
    n = n_prefix + n_suffix
    m = np.tril(np.ones((n, n), dtype=int))
    m[:n_prefix, :n_prefix] = 1
    return m

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 图片切成小块，每块一个 token")
n = num_patches(224, 14)
print(f"  224 / 14 = {224 // 14}，{224 // 14} × {224 // 14} = {n} 个图像 token；3 个相机 = {3 * n} 个")

In [ ]:
print("\n第二步 prefix-LM：前缀互相看，回答往前看")
m = prefix_lm_mask(7, 3)  # 4 个图像 token + 3 个指令 token = 前缀 7 个；回答 3 个
print(m)
print(f"  第 0 个图像 token 能看到第 6 个（指令）吗：{bool(m[0, 6])}；能看到回答吗：{bool(m[0, 7])}")

## 3. 代码里才看得到的

1. `prefix_lm_mask` 先建一个下三角（纯因果），再把左上角前缀块整块改成 1，这一步就是 prefix-LM 比纯因果多出来的那片。
2. 这里的 7 个前缀只是玩具（4 个图像 + 3 个指令）；真实 PaliGemma 是 256 + 指令长度。

## 4. 自己改一改

- 把 `num_patches` 的分辨率改成 448，看图像 token 变成多少。
- 把 `prefix_lm_mask` 的前缀长度改成 0，看它是不是退化成纯因果。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 图片切成拼图块，一块一个 token，和文字排一队；
> 眼睛 SigLIP，视神经一层线性，大脑 Gemma；前缀互相看，回答往前看。